### Focus

In this activity, you will apply Support Vector Machines (SVMs) to both a classification and a regression task using real-world datasets.

Your goals are to:
- Understand how SVM works in practice
- Explore the impact of hyperparameters
- Compare classification vs regression behavior
- Reflect on when SVMs are effective compared to ensemble methods

### Datasets
You will use:
- Wine dataset (classification)
- California Housing dataset (regression)

## Part 1 — Implementation
## Exercise 1 — SVM for Classification (Wine Dataset)

In [47]:
# Load the Wine dataset using:
from sklearn.datasets import load_wine
# This dataset contains the chemical analysis of 178 wine samples produced by 3 different cultivators.


In [48]:
# Split the dataset into training and test sets.
from sklearn.model_selection import train_test_split
wine = load_wine()
X_train, X_test, y_train, y_test = train_test_split(wine.data, wine.target, test_size=0.2, random_state=42)


In [49]:
# Scale the features (important for SVM)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [50]:
# Train an SVM classifier.
# Since SVM is fundamentally binary, use:
# One-vs-Rest (OvR), or One-vs-One (OvO)
from sklearn.svm import SVC
svm_clf = SVC(kernel='rbf', C=1, gamma='scale', decision_function_shape='ovr')
svm_clf.fit(X_train_scaled, y_train)
y_pred = svm_clf.predict(X_test_scaled) 


In [51]:
# Tune key hyperparameters:
# C 
# kernel (linear, rbf)
# gamma (if using rbf)

from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

param_grid = [
    {"kernel": ["linear"], "C": [0.1, 1, 10, 100]},
    {"kernel": ["rbf"], "C": [0.1, 1, 10, 100], "gamma": ["scale", 0.01, 0.1, 1]},
]

grid = GridSearchCV(
    estimator=SVC(decision_function_shape="ovr"),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train_scaled, y_train)

best_svm = grid.best_estimator_
y_pred = best_svm.predict(X_test_scaled)

print("Best params:", grid.best_params_)
print("Best CV accuracy:", round(grid.best_score_, 4))


Best params: {'C': 1, 'gamma': 0.01, 'kernel': 'rbf'}
Best CV accuracy: 0.9788


In [52]:
# Evaluate the model using:
# Accuracy, precision, recall
# Confusion matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="weighted")
rec = recall_score(y_test, y_pred, average="weighted")
cm = confusion_matrix(y_test, y_pred)

print("Accuracy:", round(acc, 4))
print("Precision (weighted):", round(prec, 4))
print("Recall (weighted):", round(rec, 4))
print("\nConfusion Matrix:")
print(cm)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Accuracy: 1.0
Precision (weighted): 1.0
Recall (weighted): 1.0

Confusion Matrix:
[[14  0  0]
 [ 0 14  0]
 [ 0  0  8]]
Confusion Matrix:
[[14  0  0]
 [ 0 14  0]
 [ 0  0  8]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        14
           1       1.00      1.00      1.00        14
           2       1.00      1.00      1.00         8

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36



### Questions to Answer:
- What is your best test metric to use in this case? and why?
- - best metric would be accuracy, the reason for this is this is a multiclass problem rather than binary
- Which kernel performed best?
- - "rbf"
- How did scaling affect performance?
- - Scaling improved performance, with SVM, they depend on the distance/margin so without scalling, large range features can hurt the algorithm 

## Exercise 2 — SVM for Regression (California Housing Dataset)

In [53]:
# Load the dataset using:
from sklearn.datasets import fetch_california_housing
# This dataset contains over 20,000 instances, with targets representing median house values (in hundreds of thousands of dollars).
# Because SVMs scale poorly with large datasets:
# Use a subset (~2,000 samples).

In [ ]:
# 1. Split the data
from sklearn.model_selection import train_test_split
housing = fetch_california_housing()
# Use a subset (~2,000 samples).
housing.data = housing.data[:2000]
housing.target = housing.target[:2000]

X_train, X_test, y_train, y_test = train_test_split(housing.data, housing.target, test_size=0.2, random_state=42)

In [55]:
# 2. Scale the features. 
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [56]:
# 3. Train an SVM regressor.
from sklearn.svm import SVR
svm_reg = SVR(kernel='rbf', C=1, gamma='scale')
svm_reg.fit(X_train_scaled, y_train)
y_pred = svm_reg.predict(X_test_scaled)

In [57]:
# 4. Tune hyperparameters:
# C
# epsilon
# kernel
# gamma
from sklearn.model_selection import GridSearchCV

param_grid = [
    {
        "kernel": ["linear"],
        "C": [0.1, 1, 10, 100],
        "epsilon": [0.01, 0.1, 0.5, 1.0],
    },
    {
        "kernel": ["rbf"],
        "C": [0.1, 1, 10, 100],
        "epsilon": [0.01, 0.1, 0.5, 1.0],
        "gamma": ["scale", 0.001, 0.01, 0.1, 1],
    },
]

grid = GridSearchCV(
    estimator=SVR(),
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

grid.fit(X_train_scaled, y_train)

best_svr = grid.best_estimator_
y_pred = best_svr.predict(X_test_scaled)

print("Best Params:", grid.best_params_)
print("Best CV RMSE:", round((-grid.best_score_) ** 0.5, 4))

Best Params: {'C': 10, 'epsilon': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}
Best CV RMSE: 0.4565


In [ ]:
# Evaluate using:
# RMSE (Root Mean Squared Error) and other metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("RMSE:", round(rmse, 4))
print("MSE :", round(mse, 4))
print("MAE :", round(mae, 4))
print("R^2 :", round(r2, 4))

RMSE: 0.3933
MSE : 0.1547
MAE : 0.258
R^2 : 0.8306


### Questions to Answer:
- What is your best metric? and why?
- - RMSE is the best metric here because this is regression and RMSE is in the same units as the target, so prediction error is directly interpretable, so RMSE: 0.3933 would tell me that the model is off by $39k per house but using r^2 is still a good metric here if we want to see how well the model performed
- Which kernel worked best?
- - rbf
- How sensitive was performance to C and gamma?
- -  i would say they are pretty sensitive, with best params being C = 10 and gamma = scale, this would tell me that why would be sensitive to changes.

## Part 2 — Reflection & Summary
Write a short reflection (approximately 1–2 paragraphs ):

1. What did you learn about SVM?
Consider:
- Sensitivity to scaling
- Effect of hyperparameters
- Kernel trick behavior
- Margin intuition

2. When might ensemble methods be more beneficial than SVM?
Think about:
- Large datasets
- Non-linear boundaries
- Interpretability
- Computational cost
- Stability

3. Any observations or surprises?
For example:
- Did RBF outperform linear?
- Did regression perform worse than expected?
- Was tuning expensive?
- Did SVM overfit easily?

I learned that SVM is powerful but sensitive: scaling is critical because SVM relies on distances and margins, and performance depends heavily on hyperparameters like C, gamma, and epsilon; in my regression results, the best model was SVR with kernel='rbf', C=10, epsilon=0.1, and gamma='scale', achieving RMSE=0.3933 (about $39,330, since the target is in $100,000 units) and R²=0.8306, which shows strong nonlinear fit but also confirms tuning is expensive. i also tried running my model with the full dataset, of the housing data but my computer ran along time, used only 2000 instances as the task said to but the training still took longer than the classification task. Overall, this exercise showed me that SVM can perform very well, but it requires careful preprocessing, parameter tuning, and practical limits on dataset size